# Single-field ALMA simulation

[Colab Link](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/distributed_applications_tutorials/simulation/alma_single_field_simulation.ipynb)

Port of the SIRIUS `alma_single_field` notebook: an ALMA 12 m array observation of one and then
four point sources placed by **image pixel**, simulated with an Airy-disk beam and imaged (dirty
and CLEANed) with AstroVIPER.

---
## API


In [ ]:
from astroviper.distributed_applications.simulation import simulate_processing_set

simulate_processing_set?

## Install AstroVIPER

In [ ]:
import os
from importlib.metadata import version

try:
    import astroviper  # noqa: F401

    print("Using astroviper version", version("astroviper"))
except ImportError:
    os.system("pip install --upgrade astroviper")
    import astroviper  # noqa: F401

    print("Installed astroviper version", version("astroviper"))

In [ ]:
# Set True for interactive (zoom / pan) plots via the ipympl widget backend
# (``pip install ipympl``).  Keep False for the automated notebook tests, which
# execute headless.
INTERACTIVE_PLOTS = False

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from astropy.coordinates import SkyCoord
from IPython import get_ipython

get_ipython().run_line_magic("matplotlib", "widget" if INTERACTIVE_PLOTS else "inline")

xr.set_options(display_style="html")
ARCSEC_TO_RAD = np.pi / (180 * 3600)

In [ ]:
from toolviper.dask.client import local_client

viper_client = local_client(cores=4, memory_limit="4GB")
viper_client

## ALMA 12 m array

Select the 12 m dishes of the full ALMA layout.

In [ ]:
from astroviper.utils.telescope_layout import read_telescope_layout

alma_all = read_telescope_layout("alma.all")
is_12m = alma_all.ANTENNA_DISH_DIAMETER.values == 12.0
antenna_xds = alma_all.isel(antenna_name=np.where(is_12m)[0][:40])
n_antenna = antenna_xds.sizes["antenna_name"]
print(n_antenna, "antennas")

In [ ]:
time_params = {
    "time_start": "2019-10-03T19:00:00.000",
    "time_delta": 2000.0,
    "n_samples": 18,
}
frequency_params = {
    "freq_start": 90e9,
    "freq_delta": 0.5e9,
    "n_channels": 3,
    "channel_width": 0.5e9,
    "spectral_window_name": "Band3",
}
polarization = ["XX", "YY"]

from astroviper.utils.beam_models import airy_disk_model

beam_models = [airy_disk_model("alma")]  # CASA PBMath1DAiry flavour (default)
beam_model_map = np.zeros(n_antenna, dtype=int)

## Place sources by pixel

``sin_pixel_to_celestial_coord`` converts pixel positions of a SIN-projected image (size, cell)
centred on the phase centre to sky positions.

In [ ]:
from astroviper.utils.coordinate_transforms import sin_pixel_to_celestial_coord

image_size = np.array([512, 512])
cell_size_arcsec = 0.3
cell_size = np.array([-cell_size_arcsec, cell_size_arcsec]) * ARCSEC_TO_RAD

phase_center = SkyCoord(ra="19h59m28.5s", dec="-40d44m01.5s", frame="icrs")
phase_center_ra_dec = np.array([phase_center.ra.rad, phase_center.dec.rad])[None, :]

pixels_single = np.array([[256, 256]])
point_source_ra_dec = sin_pixel_to_celestial_coord(
    phase_center_ra_dec[0], image_size, cell_size, pixels_single
)[None, :, :]
point_source_flux = np.array([1.0, 0, 0, 1.0])[None, None, None, :]

In [ ]:
result = simulate_processing_set(
    ps_store="alma_sim.ps.zarr",
    antenna_xds=antenna_xds,
    time_params=time_params,
    frequency_params=frequency_params,
    polarization=polarization,
    point_source_flux=point_source_flux,
    point_source_ra_dec=point_source_ra_dec,
    phase_center_ra_dec=phase_center_ra_dec,
    beam_models=beam_models,
    beam_model_map=beam_model_map,
    n_time_chunks=3,
    n_frequency_chunks=3,
    overwrite=True,
)
result["timing_node_tasks"][["task_id", "T_uvw", "T_visibilities", "T_write"]]

## Validate the processing set against the MSv4 schema

``xradio.schema.check.check_datatree`` checks every dataset of the processing set (coordinates,
dimensions, dtypes and attributes of the main, antenna and field/source datasets) against the
MSv4 schema.  ``simulate_processing_set`` runs this check itself (``check_schema=True``) and logs
a warning on problems; here it is run explicitly so that the result is visible.

In [ ]:
from xradio.measurement_set import open_processing_set
from xradio.schema.check import check_datatree

ps_xdt = open_processing_set("alma_sim.ps.zarr")
issues = check_datatree(ps_xdt)
print(issues)
assert str(issues) == "No schema issues found"
# the same check works on a single MSv4 (the checker dispatches on the ``type`` attribute)
print(check_datatree(ps_xdt[result["ms_name"]]))

## Dirty and cleaned images

In [ ]:
from xradio.image import load_image, make_empty_sky_image
from xradio.measurement_set import open_processing_set

from astroviper.distributed_applications.imaging import image_cube_single_field


def image_simulation(
    ps_store,
    image_store,
    image_size,
    cell_size_arcsec,
    niter=0,
    polarization_coords=("I",),
    n_chunks=2,
):
    """Make a (dirty or cleaned) cube of a simulated processing set with AstroVIPER."""
    ps_xdt = open_processing_set(ps_store)
    combined = ps_xdt.xr_ps.get_combined_field_and_source_xds()
    phase_direction = combined.FIELD_PHASE_CENTER_DIRECTION.sel(
        field_name=combined.attrs["center_field_name"]
    ).values
    image_params = {
        "image_size": list(image_size),
        "cell_size": np.array([-cell_size_arcsec, cell_size_arcsec]) * ARCSEC_TO_RAD,
        "phase_direction": phase_direction,
        "frequency_coords": ps_xdt.xr_ps.get_freq_axis().values,
        "polarization_coords": list(polarization_coords),
        "time_coords": [0],
        "fft_padding": 1.2,
        "cpp_gridder": True,
    }
    iteration_control = {
        "niter": niter,
        "nmajor": -1 if niter > 0 else 0,
        "threshold": 0.0,
        "gain": 0.1,
        "cyclefactor": 1.5,
        "cycleniter": -1,
        "minpsffraction": 0.05,
        "maxpsffraction": 0.8,
        "primary_beam_limit": 0.1,
    }
    keep = [
        "sky_residual",
        "point_spread_function",
        "primary_beam",
        "beam_fit_params_point_spread_function",
    ]
    if niter > 0:
        keep += ["sky_model", "mask"]
    image_cube_single_field(
        ps_store=ps_store,
        image_store=image_store,
        image_params=image_params,
        imaging_weights_params={
            "weighting": "natural",
            "robust": 0.5,
            "casa_weighting_implementation": True,
        },
        iteration_control_params=iteration_control,
        gridder="prolate_spheroidal",
        deconvolver="hogbom_many_threads",
        scan_intents="OBSERVE_TARGET#ON_SOURCE",
        image_data_variables_keep=keep,
        processing_set_data_group_name="base",
        single_precision_image=False,
        processing_function_threads=1,
        n_chunks=n_chunks,
        overwrite=True,
        restore=niter > 0,
        # CASA pbcor: divide the restored sky by the (power) primary beam,
        # writing SKY_RESTORED_PRIMARY_BEAM_CORRECTED (blanked with NaN below
        # the primary-beam cutoff).
        primary_beam_correction=niter > 0,
    )
    return add_sky_coordinates(load_image(image_store))


def add_sky_coordinates(img_xds):
    """Attach 2-D ``right_ascension`` / ``declination`` pixel coordinates.

    The imaging driver writes image stores without sky coordinates; rebuild them
    on the image's own grid -- SIN projection about the reference direction from
    the image metadata, cell size from the ``xr_img`` accessor -- and add them as
    non-dimensional coordinates of the (l, m) pixel axes.
    """
    img_xds.attrs["type"] = "image_dataset"  # the xr_img accessor checks this
    reference_direction = img_xds.attrs["coordinate_system_info"][
        "reference_direction"
    ]["data"]
    template = make_empty_sky_image(
        phase_center=np.asarray(reference_direction),
        image_size=[img_xds.sizes["l"], img_xds.sizes["m"]],
        cell_size=img_xds.xr_img.get_lm_cell_size(),
        frequency_coords=img_xds.frequency.values,
        pol_coords=list(img_xds.polarization.values),
        time_coords=list(img_xds.time.values),
        do_sky_coords=True,
    )
    # Assign by raw values: the template's float l / m coords differ at the
    # last bit, and aligning on them would reindex the sky coords to NaN.
    return img_xds.assign_coords(
        right_ascension=(("l", "m"), template.right_ascension.values),
        declination=(("l", "m"), template.declination.values),
    )


colorbar_labels = {
    "SKY_RESTORED": "Jy/beam",
    "SKY_RESTORED_PRIMARY_BEAM_CORRECTED": "Jy/beam",
    "SKY_RESIDUAL": "Jy/beam",
    "SKY_MODEL": "Jy/pixel",
    "POINT_SPREAD_FUNCTION": "response",
    "PRIMARY_BEAM": "power response",
    "MASK": "boolean",
}


def _sky_coordinate_axes(ax, img_xds):
    """Label a pixel-space image plot with RA / Dec on the top / right axes.

    ``load_image`` attaches the 2-D ``right_ascension`` / ``declination``
    coordinates of every pixel (the ``img_xds`` accessor machinery,
    ``do_sky_coords=True``); sample them along the image centre row / column.
    """
    from astropy import units as u
    from astropy.coordinates import Angle

    ra = img_xds.right_ascension.values  # [l, m], radians
    dec = img_xds.declination.values
    n_l, n_m = ra.shape
    top = ax.secondary_xaxis("top")
    top.set_xticks(np.linspace(0, n_l - 1, 3))
    top.set_xticklabels(
        [
            Angle(ra[int(pix), n_m // 2], u.rad).to_string(
                unit=u.hourangle, precision=1
            )
            for pix in np.linspace(0, n_l - 1, 3)
        ],
        fontsize=7,
    )
    top.set_xlabel("right ascension", fontsize=8)
    right = ax.secondary_yaxis("right")
    right.set_yticks(np.linspace(0, n_m - 1, 3))
    right.set_yticklabels(
        [
            Angle(dec[n_l // 2, int(pix)], u.rad).to_string(unit=u.deg, precision=0)
            for pix in np.linspace(0, n_m - 1, 3)
        ],
        fontsize=7,
    )
    right.set_ylabel("declination", fontsize=8)


def show_all_products(img_xds, suptitle, frequency=0, polarization=0, n_cols=4):
    """One-figure montage of every image-plane data product.

    Pixel axes on the bottom / left, RA / Dec on the top / right.
    """
    variables = [
        name for name in img_xds.data_vars if {"l", "m"} <= set(img_xds[name].dims)
    ]
    n_rows = -(-len(variables) // n_cols)
    fig, axes = plt.subplots(
        n_rows, n_cols, figsize=(4.6 * n_cols, 4.3 * n_rows), constrained_layout=True
    )
    for ax in np.ravel(axes):
        ax.set_axis_off()
    for ax, variable in zip(np.ravel(axes), variables, strict=False):
        ax.set_axis_on()
        plane = (
            img_xds[variable]
            .isel(time=0, frequency=frequency, polarization=polarization)
            .values
        )
        im = ax.imshow(plane.T, origin="lower", cmap="viridis")
        ax.set_title(variable, fontsize=10)
        ax.set_xlabel("l [pixel]")
        ax.set_ylabel("m [pixel]")
        _sky_coordinate_axes(ax, img_xds)
        fig.colorbar(im, ax=ax, shrink=0.75, label=colorbar_labels.get(variable, ""))
    fig.suptitle(
        f"{suptitle} (channel {frequency}, "
        f"{img_xds.frequency.values[frequency] / 1e9:.2f} GHz)"
    )
    return fig


def show_spectral_profiles(img_xds, pixels, flux=1.0):
    """SIRIUS-style spectrum at each source pixel.

    Top row: the apparent spectrum -- restored image value and primary beam x
    flux against frequency (they coincide; the beam narrows with frequency so
    off-centre sources show a declining apparent spectrum).  Bottom row: the
    primary-beam-corrected spectrum, which recovers the flat sky flux.
    """
    pixels = np.atleast_2d(pixels)
    frequency = img_xds.frequency.values / 1e9
    restored = img_xds.SKY_RESTORED.isel(time=0, polarization=0).values
    corrected = img_xds.SKY_RESTORED_PRIMARY_BEAM_CORRECTED.isel(
        time=0, polarization=0
    ).values
    primary_beam = img_xds.PRIMARY_BEAM.isel(time=0, polarization=0).values
    fig, axes = plt.subplots(
        2,
        len(pixels),
        figsize=(4.2 * len(pixels), 6.4),
        squeeze=False,
        constrained_layout=True,
    )
    for k, pix in enumerate(pixels):
        ax = axes[0, k]
        ax.plot(frequency, restored[:, pix[0], pix[1]], "b*-", ms=12, label="restored")
        ax.plot(
            frequency,
            primary_beam[:, pix[0], pix[1]] * flux,
            "ro-",
            fillstyle="none",
            label="primary beam x flux",
        )
        ax.set_title(f"pixel {pix}")
        ax.set_ylabel("Jy/beam" if k == 0 else None)
        ax.grid(alpha=0.3)
        if k == 0:
            ax.legend(fontsize=8)
        ax = axes[1, k]
        ax.plot(frequency, corrected[:, pix[0], pix[1]], "gs--", label="PB-corrected")
        ax.axhline(flux, color="gray", lw=0.8)
        ax.set_ylim(flux - 0.05, flux + 0.05)
        ax.set_xlabel("frequency [GHz]")
        ax.set_ylabel("Jy/beam" if k == 0 else None)
        ax.grid(alpha=0.3)
        if k == 0:
            ax.legend(fontsize=8)
    fig.suptitle("spectral profiles at the source positions")
    return fig


def show_image(
    img_xds,
    variable="SKY_RESIDUAL",
    frequency=0,
    polarization=0,
    title=None,
    vmax=None,
    colorbar_label="Jy/beam",
):
    """Plot one plane of an AstroVIPER image with l/m in arcsec."""
    plane = (
        img_xds[variable]
        .isel(time=0, frequency=frequency, polarization=polarization)
        .values
    )
    extent = (
        np.array(
            [
                img_xds.l.values[0],
                img_xds.l.values[-1],
                img_xds.m.values[0],
                img_xds.m.values[-1],
            ]
        )
        / ARCSEC_TO_RAD
    )
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(plane.T, origin="lower", extent=extent, cmap="viridis", vmax=vmax)
    ax.set_xlabel("l [arcsec]")
    ax.set_ylabel("m [arcsec]")
    ax.set_title(
        title
        or f"{variable} channel {frequency} ({img_xds.frequency.values[frequency] / 1e9:.3f} GHz)"
    )
    fig.colorbar(im, ax=ax, label=colorbar_label)
    return fig

In [ ]:
dirty = image_simulation(
    "alma_sim.ps.zarr", "alma_sim_dirty.img.zarr", image_size, cell_size_arcsec, niter=0
)
show_image(dirty, title="dirty image, single source at the phase centre")
plt.show()
clean = image_simulation(
    "alma_sim.ps.zarr",
    "alma_sim_clean.img.zarr",
    image_size,
    cell_size_arcsec,
    niter=1000,
)
print("restored peak [Jy/beam]:", clean.SKY_RESTORED.values.max())

## All imaging data products

The headline view: every image-plane product of the cleaned single-source run in one
figure (``image_data_variables_keep``).  Axes are image **pixels** (bottom / left) and
**right ascension / declination** (top / right), taken from the 2-D sky coordinates that
``load_image`` attaches to the image dataset.  ``PRIMARY_BEAM`` follows the CASA
definition: the **power** sensitivity pattern (the square of the absolute voltage
pattern), matching the ``.pb`` image of ``tclean``.  ``SKY_RESTORED_PRIMARY_BEAM_CORRECTED``
is the restored sky divided by that primary beam (CASA ``pbcor``), blanked where the beam
is below the cutoff.  The PSF Gaussian fit parameters are a table, not an image.

In [ ]:
show_all_products(clean, "single source, niter=1000")
plt.show()
clean.BEAM_FIT_PARAMS_POINT_SPREAD_FUNCTION.isel(time=0).to_series().unstack(
    "beam_params_label"
)

### Spectral profile at the source position

The SIRIUS notebooks plot the spectrum of the image at the source location (its
``display_image`` "Spectrum at source peak" panel).  With the source at the phase
centre the primary beam is 1 for every channel, so both the apparent and the
corrected spectrum are flat at 1 Jy.

In [ ]:
show_spectral_profiles(clean, pixels_single)
plt.show()

## Four sources

Sources at different offsets experience different primary-beam attenuation.

In [ ]:
pixels_multi = np.array([[256, 256], [256, 356], [156, 156], [356, 256]])
point_source_ra_dec_multi = sin_pixel_to_celestial_coord(
    phase_center_ra_dec[0], image_size, cell_size, pixels_multi
)[None, :, :]
point_source_flux_multi = np.tile(np.array([1.0, 0, 0, 1.0]), (4, 1, 1, 1))

result = simulate_processing_set(
    ps_store="alma_msource_sim.ps.zarr",
    antenna_xds=antenna_xds,
    time_params=time_params,
    frequency_params=frequency_params,
    polarization=polarization,
    point_source_flux=point_source_flux_multi,
    point_source_ra_dec=point_source_ra_dec_multi,
    phase_center_ra_dec=phase_center_ra_dec,
    beam_models=beam_models,
    beam_model_map=beam_model_map,
    n_time_chunks=3,
    n_frequency_chunks=3,
    overwrite=True,
)
clean_multi = image_simulation(
    "alma_msource_sim.ps.zarr",
    "alma_msource_clean.img.zarr",
    image_size,
    cell_size_arcsec,
    niter=2000,
)
restored = clean_multi.SKY_RESTORED.isel(time=0, frequency=0, polarization=0).values
corrected = clean_multi.SKY_RESTORED_PRIMARY_BEAM_CORRECTED.isel(
    time=0, frequency=0, polarization=0
).values
pb = clean_multi.PRIMARY_BEAM.isel(time=0, frequency=0, polarization=0).values
for pix in pixels_multi:
    power = pb[pix[0], pix[1]]
    print(
        f"pixel {pix}: restored {restored[pix[0], pix[1]]:.3f} Jy/beam, "
        f"primary beam {power:.3f}, expected flux x PB = {power:.3f} Jy/beam, "
        f"PB-corrected {corrected[pix[0], pix[1]]:.3f} Jy/beam"
    )

### Expected restored fluxes

The apparent (not primary-beam-corrected) flux of each source is the 1 Jy sky flux
attenuated by the beam of **both** antennas of every baseline: $V \propto J_1 J_2^*$,
so for identical antennas the restored peak is flux $\times$ the **power** primary
beam $P = |V|^2$ -- exactly the quantity the ``PRIMARY_BEAM`` variable stores (CASA
definition).  The simulation evaluates its per-antenna voltage patterns with the same
CASA-compatible Airy code the imager squares for its primary beam, so the expected
peaks at channel 0 (90 GHz, 10.7 m effective dish / 0.75 m blockage) are

| pixel | offset | primary beam $P$ | expected flux $\times$ $P$ |
|---|---|---|---|
| [256, 256] | 0'' | 1.000 | 1.000 Jy/beam |
| [256, 356], [356, 256] | 30'' | 0.574 | 0.574 Jy/beam |
| [156, 156] | 42.4'' | 0.306 | 0.306 Jy/beam |

and ``SKY_RESTORED_PRIMARY_BEAM_CORRECTED`` (= restored / $P$) recovers the 1 Jy sky
flux for every source.

### All data products of the four-source image\n\nThe same headline montage for the four-source image.

In [ ]:
show_all_products(clean_multi, "four sources, niter=2000")
plt.show()
residual = clean_multi.SKY_RESIDUAL.isel(time=0, frequency=0, polarization=0).values
print(
    f"residual rms {residual.std() * 1e3:.3f} mJy/beam, "
    f"peak {np.abs(residual).max() * 1e3:.3f} mJy/beam"
)

### Spectral profiles at the source positions

For the off-centre sources the primary beam narrows with frequency, so the apparent
(restored) spectrum of a flat 1 Jy source declines across the band and coincides with
the primary-beam spectrum at that pixel; the corrected spectrum is flat at 1 Jy.

In [ ]:
show_spectral_profiles(clean_multi, pixels_multi)
plt.show()

## Clean up

In [ ]:
import shutil

for path in [
    "alma_sim.ps.zarr",
    "alma_msource_sim.ps.zarr",
    "alma_sim_dirty.img.zarr",
    "alma_sim_clean.img.zarr",
    "alma_msource_clean.img.zarr",
]:
    shutil.rmtree(path, ignore_errors=True)
viper_client.close()